Assess how well top-ranked ligands can predict a gene set of interest
================

This tutorial assesses the ligands prioritized by NicheNet in their
ability to predict a gene set of interest. We will first follow the
steps of [Perform NicheNet analysis starting from an AnnData
object](wrapper.ipynb)) to obtain ligands rankings. Make sure you
understand the steps and output of a basic NicheNet analysis (more
information in [Perform NicheNet analysis starting from an AnnData object:
step-by-step analysis](steps.ipynb). You can also apply this
tutorial to the [NicheNet’s ligand activity analysis on a gene set of
interest](ligand_activity_gene_set.ipynb) notebook.


In [1]:
from nichenetpy.wrappers import run_nichenet
from nichenetpy.gene_symbol import mouse_alias_info

import anndata
import os
import requests
import pickle

Download the model pickle

In [2]:
filename = "nichenet_mouse.pkl"
file_path = os.path.join("./tutorial_files", filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14887637/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Download the AnnData object

In [3]:
data_path = os.path.normpath("./tutorial_files/AnnData")
if not os.path.exists(data_path):
    os.makedirs(data_path)
filename = "annData3531889.h5"
file_path = os.path.join(data_path, filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14859451/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Perform nichenet analysis using the wrapper function

In [4]:
ann = anndata.io.read_h5ad(os.path.join(data_path, "annData3531889.h5"))
ann.var_names = ann.var["gene"]
mouse_alias_info.alias_to_symbol(ann)
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())
predictor = model["predictor"]
lr_network = model["lr_network"]
lr_sig = model["lr_sig"]
receiver = "CD8 T"
sender_celltypes = ["CD4 T","Treg", "Mono", "NK", "B", "DC"]
output = run_nichenet(
    ann,
    predictor,
    lr_network,
    lr_sig=lr_sig,
    receiver=receiver,
    sender_celltypes=sender_celltypes,
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    get_expressed_genes_pct=0.05
)
geneset_oi = output["geneset_oi"]
expressed_genes_receiver = output["expressed_genes_receiver"]
ligands_oi = output["best_upstream_ligands"]

## Assess how well top-ranked ligands can predict a gene set of interest

For the top 30 ligands, we will now build a multi-ligand model that uses
all top-ranked ligands to predict whether a gene belongs to the gene set
of interest (differentially expressed genes in CD8 T cells after LCMV
infection) or not. This classification model will be trained via
cross-validation and returns a probability for every gene.

In [5]:
from nichenetpy.prediction import assess_rf_class_probabilities
n = 2
k = 3
gene_predictions_top30_list = [
    assess_rf_class_probabilities(
        folds=k,
        geneset=geneset_oi,
        background_expressed_genes=expressed_genes_receiver,
        ligands_oi=ligands_oi,
        predictor=predictor,
    ) for _ in range(n)
]

In [6]:
gene_predictions_top30_list[0].sort_values(by="prediction", ascending=False)

,gene,response,prediction
78,Gbp8,1,0.987
59,H2-T10,1,0.984
170,Trim12a,0,0.963
73,Trim30a,1,0.963
45,Ddx60,1,0.962
...,...,...,...
507,Safb,0,0.000
499,Polr2g,0,0.000
780,Gnptg,0,0.000
118,Araf,0,0.000
